# 02 Clean

One section per modality. Each section reads what `01_acquire` saved, cleans it, and saves a cleaned table to `data/clean/`.

# All imports

In [2]:
import pandas as pd
import re

---
# 1. API: TheirStack postings

Load the TheirStack postings saved by 01_acquire and check what's missing

In [12]:
postings = pd.read_csv("../data/clean/theirstack_postings.csv")
print("postings:", postings.shape)
print(postings.isna().sum())

postings: (149, 21)
id                     0
job_title              0
normalized_title     149
company                0
location              12
state_code             2
latitude               1
longitude              1
date_posted            0
seniority              0
remote                 0
hybrid                 0
min_annual_salary    104
max_annual_salary    104
salary_currency      115
salary_string        116
technology_slugs       0
description            0
url                    0
source_file            0
search_role            0
dtype: int64


Remove postings that aren't data jobs and give each posting a clean role

In [13]:
# 1. Drop titles that matched "data" but aren't data jobs
not_data = ["data cent", "electrical", "mechanical", "security engineer", "data security",
            "interchange", "test data quality", "quality engineer iii"]

is_not_data = postings["job_title"].str.lower().str.contains("|".join(not_data))
print("Removed", is_not_data.sum(), "non-data postings:")
print(postings.loc[is_not_data, "job_title"].to_string())

before = len(postings)
postings = postings[~is_not_data]
print(before, "->", len(postings))

# 2. Drop repeat postings (same title at the same company)
before = len(postings)
postings = postings.drop_duplicates(subset=["job_title", "company"])
print(before, "->", len(postings), "after removing repeats")

# 3. Clean role column
postings["role"] = postings["search_role"]
print(postings["role"].value_counts())

Removed 12 non-data postings:
13                  Electronic Data Interchange Analyst
50                        Senior Data Security Engineer
51             Senior Security Engineer (Data Security)
56                        Senior Data Security Engineer
65    Ingénieur(e) mécanique des Centre de Données /...
71             Senior Security Engineer (Data Security)
72    Ingénieur(e) électrique de centres de données ...
73                           Test Data Quality Engineer
83                           Test Data Quality Engineer
85                   Quality Engineer III (Data domain)
91    Electrical Engineer-Data Centre Design & Planning
98    Electrical Engineer-Data Centre Design & Planning
149 -> 137
137 -> 124 after removing repeats
role
data scientist    46
data analyst      42
data engineer     36
Name: count, dtype: int64


Tidy the location: province code and a remote yes/no column

In [14]:
import numpy as np

# 1. Drop the US-only posting
before = len(postings)
postings = postings[postings["state_code"] != "NY"]
print(before, "->", len(postings), "after removing US-only posting")

# 2. Province: use state_code, but "00" means unknown, so try the location text instead
province_names = {"British Columbia": "BC", "Ontario": "ON", "Quebec": "QC", "Alberta": "AB"}
postings["province"] = postings["state_code"].replace("00", np.nan)
for name, code in province_names.items():
    found = postings["province"].isna() & postings["location"].str.contains(name, na=False)
    postings.loc[found, "province"] = code

# 3. Blank out the placeholder "centre of Canada" coordinates
placeholder = (postings["latitude"].round(2) == 60.11) & (postings["longitude"].round(2) == -113.64)
print(placeholder.sum(), "postings had placeholder coordinates, set to blank")
postings.loc[placeholder, ["latitude", "longitude"]] = np.nan

# 4. Remote: also count locations tagged {{REMOTE}}
postings["remote"] = postings["remote"] | postings["location"].str.contains("REMOTE", na=False)

print(postings["province"].value_counts(dropna=False))
print("remote:", postings["remote"].sum(), "| hybrid:", postings["hybrid"].sum(),
      "| with real coordinates:", postings["latitude"].notna().sum())

124 -> 123 after removing US-only posting
21 postings had placeholder coordinates, set to blank
province
ON     73
NaN    19
QC     14
AB      8
BC      7
PE      2
Name: count, dtype: int64
remote: 28 | hybrid: 38 | with real coordinates: 101


Turn salary into one annual CAD column

In [15]:
usd_to_cad = 1.40   # USD/CAD rate 

has_salary = postings["min_annual_salary"].notna()
postings["salary_currency"] = postings["salary_currency"].fillna("CAD").where(has_salary)

midpoint = (postings["min_annual_salary"] + postings["max_annual_salary"]) / 2
postings["salary_annual_cad"] = np.where(postings["salary_currency"] == "USD", midpoint * usd_to_cad, midpoint).round(0)

print(has_salary.sum(), "of", len(postings), "postings have a salary")
print(postings["salary_currency"].value_counts())
print(postings.groupby("role")["salary_annual_cad"].agg(["count", "median"]))

38 of 123 postings have a salary
salary_currency
CAD    26
USD    12
Name: count, dtype: int64
                count    median
role                           
data analyst       11  122500.0
data engineer      12  135100.0
data scientist     15  139800.0


Group seniority into junior, mid and senior

In [16]:
postings["seniority"] = postings["seniority"].replace({"mid_level": "mid", "staff": "senior"})
print(postings["seniority"].value_counts(dropna=False))
print(pd.crosstab(postings["role"], postings["seniority"]))

seniority
mid       76
senior    45
junior     2
Name: count, dtype: int64
seniority       junior  mid  senior
role                               
data analyst         2   28      12
data engineer        0   24      11
data scientist       0   24      22


Make one True/False column per skill by searching the description

In [17]:
skills = {
    "sql": r"\bsql\b",
    "python": r"\bpython\b",
    "r": r"\br\b(?!&)",
    "excel": r"\bexcel\b",
    "power_bi": r"\bpower ?bi\b",
    "tableau": r"\btableau\b",
    "spark": r"\b(?:py)?spark\b",
    "airflow": r"\bairflow\b",
    "dbt": r"\bdbt\b",
    "aws": r"\baws\b|amazon web services",
    "azure": r"\bazure\b",
    "gcp": r"\bgcp\b|google cloud",
    "snowflake": r"\bsnowflake\b",
    "databricks": r"\bdatabricks\b",
    "machine_learning": r"machine learning|\bml\b",
    "statistics": r"\bstatistic",
}

text = postings["description"].fillna("").str.lower()
for skill, pattern in skills.items():
    postings["skill_" + skill] = text.str.contains(pattern, regex=True)

skill_cols = ["skill_" + s for s in skills]
print((postings.groupby("role")[skill_cols].mean() * 100).round(0).T)

role                    data analyst  data engineer  data scientist
skill_sql                       76.0           83.0            70.0
skill_python                    60.0           91.0            89.0
skill_r                         12.0            0.0            20.0
skill_excel                     26.0            3.0             9.0
skill_power_bi                  52.0           20.0             9.0
skill_tableau                   29.0           11.0            11.0
skill_spark                     17.0           60.0            24.0
skill_airflow                    5.0           17.0             0.0
skill_dbt                       14.0           20.0             4.0
skill_aws                       10.0           43.0            28.0
skill_azure                     21.0           60.0            30.0
skill_gcp                       10.0           23.0            26.0
skill_snowflake                 21.0           31.0             4.0
skill_databricks                19.0           5

Save the cleaned postings to data/clean/

In [19]:
role_to_noc = {"data analyst": "21223", "data engineer": "21211", "data scientist": "21211"}
postings["noc"] = postings["role"].map(role_to_noc)

keep = ["id", "role", "noc", "job_title", "company", "seniority",
        "location", "province", "latitude", "longitude", "remote", "hybrid",
        "date_posted", "salary_annual_cad", "salary_currency"] + skill_cols + ["url"]

postings_clean = postings[keep]
postings_clean.to_csv("../data/clean/postings_clean.csv", index=False)
print(postings_clean.shape)
postings_clean.head()

(123, 32)


,id,role,noc,job_title,company,seniority,location,province,latitude,longitude,...,skill_airflow,skill_dbt,skill_aws,skill_azure,skill_gcp,skill_snowflake,skill_databricks,skill_machine_learning,skill_statistics,url
0,966358831,data analyst,21223,"Product Data Analyst, eCommerce [AQ-27829]",Aquent,mid,"Toronto, Ontario",ON,43.706429,-79.398643,...,False,False,False,False,False,False,False,False,False,https://www.linkedin.com/jobs/view/4492343389/
1,965542882,data analyst,21223,Data Analyst (Revenue & Margin),Trulioo,mid,"Vancouver, British Columbia, Canada {{REMOTE}}",BC,NaN,NaN,...,True,True,False,False,False,False,False,False,True,https://jobs.ashbyhq.com/trulioo/775fe1fa-c4a9...
2,965542881,data analyst,21223,Senior Data Analyst (Revenue & Margin),Trulioo,senior,"Vancouver, British Columbia, Canada {{REMOTE}}",BC,NaN,NaN,...,True,True,False,False,False,False,False,False,True,https://jobs.ashbyhq.com/trulioo/2755d9fa-1fa6...
3,967529933,data analyst,21223,"Business Analyst Co-op, Data Governance",Teck Resources Limited,junior,"Vancouver, British Columbia, CA",BC,49.249660,-123.119339,...,False,False,False,False,False,False,False,False,False,https://jobs.teck.com/job/Vancouver-Business-A...
4,962244502,data analyst,21223,"Analyst, Data & Business Intelligence (Temporary)",Bombardier,mid,"Dorval, QC H4S 2A3",QC,45.479332,-73.726685,...,False,False,False,False,False,False,True,True,False,https://jobs.bombardier.com/job/Dorval-Analyst...


---
# 2. Web scraping: Job Bank wages

Load the Job Bank wages saved by 01_acquire and check what's missing

In [4]:
wages = pd.read_csv("../data/clean/jobbank_wages_by_region.csv")
print("wages:", wages.shape)
print(wages.isna().sum())

wages: (255, 8)
level            0
role             0
noc              0
province         0
region           0
wage_low       136
wage_median    136
wage_high      136
dtype: int64


Turn hourly wages into annual wages and keep region and province rows apart

In [18]:
hours_per_year = 1950   # 37.5 hours/week x 52 weeks

for col in ["wage_low", "wage_median", "wage_high"]:
    wages[col + "_annual"] = (wages[col] * hours_per_year).round(0)

wages["noc"] = wages["noc"].astype(str)

print(wages.groupby(["level", "role"])["wage_median"].apply(lambda s: s.notna().sum()))
wages[wages["level"] == "province"][["role", "province", "wage_median", "wage_median_annual"]].head(13)

level     role          
province  data analyst      10
          data engineer      9
          data scientist     9
region    data analyst      39
          data engineer     26
          data scientist    26
Name: wage_median, dtype: int64


,role,province,wage_median,wage_median_annual
0,data analyst,NL,43.75,85312.0
5,data analyst,PE,42.37,82622.0
6,data analyst,NS,40.85,79658.0
12,data analyst,NB,40.87,79696.0
18,data analyst,QC,40.38,78741.0
36,data analyst,ON,41.03,80008.0
48,data analyst,MB,44.71,87184.0
57,data analyst,SK,44.40,86580.0
64,data analyst,AB,38.46,74997.0
73,data analyst,BC,40.38,78741.0


Save the cleaned wages to data/clean/

In [20]:
wages_clean = wages[["role", "noc", "level", "province", "region",
                     "wage_low", "wage_median", "wage_high",
                     "wage_low_annual", "wage_median_annual", "wage_high_annual"]]
wages_clean.to_csv("../data/clean/wages_clean.csv", index=False)
print(wages_clean.shape)

(255, 11)


---
# 3. Database / SQL: Job Bank open data

---
# 4. Technique beyond class (to be decided)